In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# ==============================================================================
# 1. Carregamento e Mapeamento (Mantendo sua lógica original)
# ==============================================================================
file_path = 'TELESAPFormulrioMdic-EXPORTAOBOLSISTAAtua_DATA_LABELS_2025-07-23_1329.csv'
df = pd.read_csv(file_path, low_memory=False)

col_escolaridade = 'Escolaridade'

cols_inf = [c for c in df.columns if 'doenças infecciosas' in c.lower() and 'choice=' in c.lower() and 'não' not in c.lower() and 'outras' not in c.lower()]
cols_cro = [c for c in df.columns if 'doenças crônicas' in c.lower() and 'choice=' in c.lower() and 'não' not in c.lower() and 'outras' not in c.lower()]
cols_esp = [c for c in df.columns if ('encaminhamento' in c.lower() or 'especialidade' in c.lower()) and 'choice=' in c.lower()]
col_status = next((c for c in df.columns if any(x in c.lower() for x in ['desfecho', 'status', 'atendimento'])), None)

# ==============================================================================
# 2. Tratamento Longitudinal e Filtros
# ==============================================================================
df = df.sort_values(['Record ID', 'Repeat Instance'], na_position='first')
cols_to_fill = [col_escolaridade] + cols_inf + cols_cro
df[cols_to_fill] = df.groupby('Record ID')[cols_to_fill].ffill()

df_ev = df[df['Repeat Instrument'].notna()].copy()
df_ev = df_ev.dropna(subset=[col_escolaridade])
df_ev = df_ev[~df_ev[col_escolaridade].astype(str).str.contains('Ignorado', case=False)]

# ==============================================================================
# 3. Cálculo das Métricas
# ==============================================================================
df_ev['Carga Infecciosas'] = (df_ev[cols_inf] == 'Checked').sum(axis=1)
df_ev['Carga Crônicas'] = (df_ev[cols_cro] == 'Checked').sum(axis=1)

if col_status:
    df_ev['Absenteísmo (%)'] = df_ev[col_status].astype(str).str.contains('não compareceu|falta', case=False, na=False).astype(int) * 100
else:
    df_ev['Absenteísmo (%)'] = 0

df_ev['Especialidades (%)'] = (df_ev[cols_esp] == 'Checked').any(axis=1).astype(int) * 100

# ==============================================================================
# 4. GERAÇÃO DA BASE CONSOLIDADA (A TABELA PARA O ARTIGO)
# ==============================================================================
metricas = ['Carga Infecciosas', 'Carga Crônicas', 'Absenteísmo (%)', 'Especialidades (%)']
resumo_stats = []

for m in metricas:
    # Agrupa por escolaridade e calcula estatísticas básicas
    stats_m = df_ev.groupby(col_escolaridade)[m].agg(['mean', 'std', 'count']).reset_index()
    
    # Cálculo do Erro Padrão (SEM) e Intervalo de Confiança (95%)
    stats_m['sem'] = stats_m['std'] / np.sqrt(stats_m['count'])
    stats_m['ci_95_margin'] = stats_m['sem'] * 1.96
    stats_m['metrica'] = m
    
    resumo_stats.append(stats_m)

# Unifica tudo em uma tabela pivotada para fácil leitura
df_base_visualizacao = pd.concat(resumo_stats)

# Salva a base processada (linha a linha) e a consolidada (estatísticas)
df_ev.to_csv('base_processada_artigo.csv', index=False, encoding='utf-8-sig')
df_base_visualizacao.to_csv('tabela_descritiva_escolaridade.csv', index=False, encoding='utf-8-sig')

# ==============================================================================
# 5. Visualização e Relatório (Mantendo seu estilo)
# ==============================================================================
ordem_esc = df_ev.groupby(col_escolaridade)['Carga Infecciosas'].mean().sort_values(ascending=False).index

plt.figure(figsize=(16, 12))
sns.set_theme(style="whitegrid")

cores = ['#c0392b', '#2980b9', '#8e44ad', '#27ae60']

for i, m in enumerate(metricas):
    plt.subplot(2, 2, i+1)
    sns.pointplot(data=df_ev, x=col_escolaridade, y=m, order=ordem_esc, color=cores[i], join=False, capsize=.15, errorbar=('ci', 95))
    
    grupos = [g[m].values for n, g in df_ev.groupby(col_escolaridade) if len(g) > 1]
    f_stat, p_val = stats.f_oneway(*grupos) if len(grupos) > 1 else (0, 1)
    
    plt.title(f"{m}\n(ANOVA p: {p_val:.4f})", fontsize=13, fontweight='bold')
    plt.xticks(rotation=45, ha='right')

plt.tight_layout()
plt.show()

print(f"\n{'='*80}\nBASE DE DADOS GERADA COM SUCESSO\n{'='*80}")
print(f"1. 'base_processada_artigo.csv' (Dados individuais tratados)")
print(f"2. 'tabela_descritiva_escolaridade.csv' (Médias e IC 95% do gráfico)")